# 맵퍼 워크벤치

**두 줄:** 표 이름(또는 샘플 파일)을 적으면 운영이 맵퍼에 넘기는 모양 그대로 DataFrame 이 온다.
자유폼 셀에서 만들고, 마지막 셀이 `@mapper` 함수 파일을 만든다.

> 🔴 **입력은 운영의 구성 그대로다.** `outbox_expand` 가 접힌 이벤트의 행을 다시 payload 로
> 읽는 그 두 함수를 지나 `payloads_to_df` 로 온다 — 봉투를 여기서 새로 조립하면 운영이 절대
> 만들지 않는 프레임 위에서 맵퍼를 쓰게 된다.

> ⚠️ **표 이름을 쓰면 DB 가 필요하다**(서버가 강제하는 «읽기 전용» 커넥션). 안 열리면 샘플 파일
> 경로를 주면 된다 — 같은 모양이 나온다.

## 0. 부트스트랩

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# 저장소 뿌리를 «찾는다» — 이 박스의 절대경로를 박지 않는다.
NB_DIR = Path.cwd()
REPO = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "server" / "dev_bench.py").is_file())
SERVER = REPO / "server"
if str(SERVER) not in sys.path:
    sys.path.insert(0, str(SERVER))

import pandas as pd

import dev_bench

# 🔴 커널을 확인한다. env 가 아닌 python 으로 돌면 psycopg2 가 없어 DB 셀이 «조용히» 못 연다.
print("interpreter :", sys.executable)
print("repo        :", REPO)
print("pandas      :", pd.__version__)

## 1. 내가 고를 것 — **이 셀만 고친다**

In [ ]:
SOURCE = "inventory_master"    # 표 이름, 또는 샘플 파일 경로
ROWS   = 20                    # 표에서 가져올 행 수 (row_ids=[...] / where="..." 도 된다)
NAME   = "my_mapper"           # 발행할 맵퍼 이름 = chain_rules.json 이 부르는 이름
PARAMS = ()                    # 규칙이 이 맵퍼에 넘길 params 이름들

print("source :", SOURCE)
print("name   :", NAME, " params:", PARAMS or "없음")

## 2. 입력 — **운영이 맵퍼에 넘기는 모양 그대로**

In [ ]:
DF = dev_bench.input_for_mapper(SOURCE, rows=ROWS)

print(f"{len(DF):,} rows x {len(DF.columns)} cols")
print("columns:", list(DF.columns))
DF.head()

## 3. 만들기 — **자유폼**

인자 이름이 `(df, db)` 인 것은 우연이 아니다: 이 본문이 그대로 `@mapper` 함수의 본문이 된다.
`db` 는 읽기 전용 세션(또는 `None`)이고, 읽을 것이 있으면 `mapper_sdk.sql(db, "...", {...})`.

⚠️ **컬럼 이름은 «읽는다», 적지 않는다.** 그리고 돌려주는 것은 DataFrame 이다 — 업서트 봉투와
비즈니스 키는 데코레이터가 붙인다.

In [ ]:
def build(df, db):
    out = df.copy()
    return out


OUT = build(DF, None)
print(f"{len(OUT):,} rows x {len(OUT.columns)} cols")
OUT.head()

## 4. 발행 — 그리고 **같은 프레임으로 다시 채점한다**

🔴 옮겨 적지 않는다. `publish_mapper` 는 파일을 쓴 «뒤» 그 파일을 다시 import 해 **같은 `DF`**
위에서 돌리고 `OUT` 과 대조한다. 다르면 파일을 지우고 이름을 대어 거절한다.

⚠️ 같은 이름의 파일이 이미 있으면 **거절**한다(덮어쓰기 인자는 없다) — 누군가 그 파일을 돌리고
있고, 노트북 셀에서 하는 배포는 아무도 부탁한 적이 없다.

In [ ]:
published = dev_bench.publish_mapper(
    NAME,
    PARAMS,
    dev_bench.cell_body(build),
    frame=DF,
    expected=OUT,
)
print("작성 :", published["path"])
print("행   :", f"{len(published['rows']):,}")

## 5. 시험으로 남기기 (선택)

폴더 «하나»가 시험 하나다 — `input.*` + 맵퍼 파일 + `rule.json` + `expected.tsv`.
아래를 돌리면 `server/tests/samples/mapper/<NAME>/` 이 생기고, 그 뒤로는 pytest 가 매번 채점한다.

In [ ]:
import json
import shutil

folder = SERVER / "tests" / "samples" / "mapper" / NAME
folder.mkdir(parents=True, exist_ok=True)

DF.to_csv(folder / "input.csv", index=False, encoding="utf-8")
(folder / "rule.json").write_text(
    json.dumps({"mapper": NAME, "target_table": "bench_target"}, indent=2), encoding="utf-8")
shutil.copy(published["path"], folder / f"{NAME}.py")
(folder / "expected.tsv").write_text(
    dev_bench.rows_to_tsv(published["rows"]), encoding="utf-8")

print("시험 폴더:", folder)
print(dev_bench.rows_to_tsv(published["rows"])[:400])